# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhardwaj-ayush03/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule in plain words:** review first the pages that get a lot of search exposure but have not been updated in a while. Score = 0.6 x visibility + 0.4 x time since last update, both from observable columns only (`impressions_90d`, `days_since_last_update`). I do not use `trend_direction` or `trend_pct`, because they define the label.

**Reason code (only one):** `visible_not_recently_updated` = `impressions_90d >= 500` and `days_since_last_update >= 90`. **Action label:** `refresh_review` if the code fires, otherwise `monitor`.

**Signal check 1: staleness (behind the refresh flags): MIXED.** Pages 90-179 days since update are labelled declining more often (61.1%, n = 9,171) than fresh pages under 90 days (51.2%, n = 20,655), and the base rate is 54.2%. But pages 180+ days are *below* the base rate (47.1%, n = 174), so "the staler, the worse" is not supported. I use the 90-day cut and cap the staleness credit at 180 days, so the rule does not lean on the small 180+ group.

**Signal check 2: volume (behind quick-win): CONFIRMED.** Pages with under 10 impressions are rarely labelled declining (20.1%, n = 3,746), while pages with 100+ impressions sit around 58-62% (n = 5,280 / 6,511 / 10,215). Low-volume pages barely have room to decline, so visibility is a fair condition for spending review time.

**Caveat:** both checks use the current-window label `trend_direction == "down"` as a proxy, so they are directional, not proof.

In [1]:
from pathlib import Path
import subprocess, json
import numpy as np
import pandas as pd

CSV = "data/raw/content_refresh_anonymized.csv"
root = next((Path(p) for p in [".", "..", "../.."] if (Path(p) / CSV).exists()), None)
if root is None:  # e.g. Colab opened straight from GitHub: pull the public starter data
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/flyrank-bih/flyrank-ml-internship-starter", "_starter"], check=True)
    root = Path("_starter")
raw = pd.read_csv(root / CSV)

pages = (raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)]
         .drop_duplicates("content_id").copy())
# the label is used ONLY to check signals and evaluate; it never enters the score
pages["declining"] = pages["trend_direction"].str.lower().eq("down").astype(int)
base = pages["declining"].mean()
print(f"pages: {len(pages)} | base declining rate: {base:.3f}\n")

def bucket_table(col, bins, labels):
    b = pd.cut(pages[col], bins, right=False, labels=labels)
    return pages.groupby(b, observed=True)["declining"].agg(n="size", declining_rate="mean").round(3)

print("SIGNAL 1: days since last update (staleness, behind the refresh flags)")
print(bucket_table("days_since_last_update", [0, 90, 180, np.inf],
                   ["<90 days", "90-179 days", "180+ days"]), "\n")

print("SIGNAL 2: impressions in 90 days (volume / visibility, behind quick-win)")
print(bucket_table("impressions_90d", [0, 10, 100, 500, 2000, np.inf],
                   ["<10", "10-99", "100-499", "500-1999", "2000+"]))

pages: 30000 | base declining rate: 0.542

SIGNAL 1: days since last update (staleness, behind the refresh flags)
                            n  declining_rate
days_since_last_update                       
<90 days                20655           0.512
90-179 days              9171           0.611
180+ days                 174           0.471 

SIGNAL 2: impressions in 90 days (volume / visibility, behind quick-win)
                     n  declining_rate
impressions_90d                       
<10               3746           0.201
10-99             4248           0.555
100-499           5280           0.604
500-1999          6511           0.618
2000+            10215           0.581


## 2. Build the ranked queue (writes the CSV)

The score, reason code and action label are computed from observable inputs only. The queue is written to `work/outputs/baseline_action_score.csv` (kept out of git by design). Precision@K is computed afterwards, against the label, as a reference point for my Week-5 model to beat.

In [2]:
# Rule: visible pages that have not been updated recently get reviewed first.
vis = (np.log10(pages["impressions_90d"].clip(lower=1)) / np.log10(20000)).clip(0, 1)
vis = vis.where(pages["impressions_90d"] >= 100, 0)              # below 100 impressions = no visibility credit
stale = (pages["days_since_last_update"].clip(0, 180) / 180)      # capped so one very old page cannot dominate
pages["baseline_score"] = (100 * (0.6 * vis + 0.4 * stale)).round(2)

fires = (pages["impressions_90d"] >= 500) & (pages["days_since_last_update"] >= 90)
pages["reason_code"] = np.where(fires, "visible_not_recently_updated", "none")
pages["action_label"] = np.where(fires, "refresh_review", "monitor")

queue = pages.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
out_cols = ["rank", "content_id", "client_id", "baseline_score", "reason_code", "action_label",
            "impressions_90d", "days_since_last_update", "avg_position", "ctr", "word_count"]
Path("work/outputs").mkdir(parents=True, exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

def p_at(k): return round(float(queue["declining"].head(k).mean()), 3)
metrics = {"n_pages": int(len(queue)), "base_declining_rate": round(float(base), 3),
           "precision_at_20": p_at(20), "precision_at_50": p_at(50),
           "n_reason_code_fires": int(fires.sum()),
           "rule": "0.6*visibility + 0.4*staleness (observable inputs only)"}
Path("work/outputs/w04_baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
print("\nreason codes:\n", queue["reason_code"].value_counts())
queue[out_cols].head(10)

{
  "n_pages": 30000,
  "base_declining_rate": 0.542,
  "precision_at_20": 0.85,
  "precision_at_50": 0.58,
  "n_reason_code_fires": 6575,
  "rule": "0.6*visibility + 0.4*staleness (observable inputs only)"
}

reason codes:
 reason_code
none                            23425
visible_not_recently_updated     6575
Name: count, dtype: int64


,rank,content_id,client_id,baseline_score,reason_code,action_label,impressions_90d,days_since_last_update,avg_position,ctr,word_count
0,1,content_1bfaa38ff26c,client_7f2253d7e2,100.00,visible_not_recently_updated,refresh_review,25715,194,22.2,0.23,3861.0
1,2,content_7368877ea310,client_7f2253d7e2,100.00,visible_not_recently_updated,refresh_review,59472,194,24.8,0.13,2591.0
2,3,content_cf56e2e2e282,client_7f2253d7e2,100.00,visible_not_recently_updated,refresh_review,61678,194,19.7,0.15,5125.0
3,4,content_0a91db491d14,client_7f2253d7e2,97.53,visible_not_recently_updated,refresh_review,13299,193,10.5,0.49,3478.0
4,5,content_5feee3994adb,client_7f2253d7e2,94.30,visible_not_recently_updated,refresh_review,7812,194,39.0,0.01,3590.0
5,6,content_c2d929d83eaa,client_7f2253d7e2,94.10,visible_not_recently_updated,refresh_review,7558,193,17.9,0.20,4758.0
6,7,content_cb7e312f5d32,client_9f14025af0,93.56,visible_not_recently_updated,refresh_review,21272,151,12.6,2.45,1108.0
7,8,content_b16bd7307b39,client_7f2253d7e2,91.08,visible_not_recently_updated,refresh_review,4590,194,31.0,0.00,4329.0
8,9,content_fe16a55cd13d,client_7f2253d7e2,91.04,visible_not_recently_updated,refresh_review,4556,194,16.4,0.33,3388.0
9,10,content_ecb6215e79fd,client_7f2253d7e2,90.87,visible_not_recently_updated,refresh_review,4429,194,25.3,0.38,4486.0


## 3. Top-10 review

For each of my top ten: the action, why it is there (its own numbers), and what would make it wrong. The "what would make it wrong" line depends on that page's own values (position, CTR, word count). `outcome_check` is only a look at the label afterwards and was not used in the score.

In [3]:
def wrong_if(r):
    if r["avg_position"] > 20:
        return "it sits beyond position 20, so the real problem may be ranking or intent, not staleness"
    if r["ctr"] >= 2:
        return "CTR is already strong, so the page may be healthy and an edit could hurt"
    if r["word_count"] < 800:
        return "the page is thin, so it may need a rewrite or a merge rather than a refresh"
    return "the drop is seasonal or the update date is not a real content change"

top10 = queue.head(10).copy()
top10["why"] = top10.apply(lambda r: f"{int(r['impressions_90d']):,} impressions and {int(r['days_since_last_update'])} days since update (score {r['baseline_score']})", axis=1)
top10["what_would_make_it_wrong"] = top10.apply(wrong_if, axis=1)
top10["outcome_check"] = np.where(top10["declining"] == 1, "labelled declining", "not labelled declining")
top10[["rank", "action_label", "why", "what_would_make_it_wrong", "outcome_check"]]

,rank,action_label,why,what_would_make_it_wrong,outcome_check
0,1,refresh_review,"25,715 impressions and 194 days since update (...","it sits beyond position 20, so the real proble...",labelled declining
1,2,refresh_review,"59,472 impressions and 194 days since update (...","it sits beyond position 20, so the real proble...",labelled declining
2,3,refresh_review,"61,678 impressions and 194 days since update (...",the drop is seasonal or the update date is not...,labelled declining
3,4,refresh_review,"13,299 impressions and 193 days since update (...",the drop is seasonal or the update date is not...,labelled declining
4,5,refresh_review,"7,812 impressions and 194 days since update (s...","it sits beyond position 20, so the real proble...",labelled declining
5,6,refresh_review,"7,558 impressions and 193 days since update (s...",the drop is seasonal or the update date is not...,labelled declining
6,7,refresh_review,"21,272 impressions and 151 days since update (...","CTR is already strong, so the page may be heal...",not labelled declining
7,8,refresh_review,"4,590 impressions and 194 days since update (s...","it sits beyond position 20, so the real proble...",labelled declining
8,9,refresh_review,"4,556 impressions and 194 days since update (s...",the drop is seasonal or the update date is not...,labelled declining
9,10,refresh_review,"4,429 impressions and 194 days since update (s...","it sits beyond position 20, so the real proble...",labelled declining


## 4. Weak picks + leakage check

**Weak picks:** 9 of my top 10 come from one client, and those pages share almost the same `days_since_last_update` (193-194), which looks like a site-wide or bulk update date, not a page-level edit. The top 50 come from only 6 clients, with the largest holding 40% of them. So the rule may be measuring a client's update habit as much as page staleness. Several of the top pages also sit beyond position 20 or have very low CTR, where a refresh may not be the right fix.

**Leakage check:** the score uses only `impressions_90d` and `days_since_last_update`. `trend_direction` and `trend_pct` (which define the label) are not inputs, and no future window is used. The label appears only in the signal tables and in the Precision@K evaluation.

**Directional only:** the signal tables and the label come from the same sample, so Precision@K is a reference point, not a held-out result. My Week-5 model should be compared to this baseline on held-out clients.

In [4]:
top50 = queue.head(50)
print("clients in top 50:", top50["client_id"].nunique())
print(top50["client_id"].value_counts().head(3))
print("share of top 50 from the largest client:", round(top50["client_id"].value_counts(normalize=True).iloc[0], 2))
print("days_since_last_update in top 10:", queue.head(10)["days_since_last_update"].tolist())
print("label-derived columns in the score inputs:", {"trend_direction", "trend_pct"} & {"impressions_90d", "days_since_last_update"})

clients in top 50: 8
client_id
client_7f2253d7e2    15
client_6208ef0f77    15
client_19581e27de    12
Name: count, dtype: int64
share of top 50 from the largest client: 0.3
days_since_last_update in top 10: [194, 194, 194, 193, 194, 193, 151, 194, 194, 194]
label-derived columns in the score inputs: set()


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.